In [1]:
import numpy as np

import transpose_invariance as tpi

import skimage as ski
import skimage.filters.ridges as sfr

/Volumes/zorg/mb312/dev_trees/coordinate-review/main/src/skimage/feature/peak.py:10: ExperimentalAPIWarning: Importing from the `skimage2` namespace is experimental. Its API is under development and considered unstable!
  import skimage2 as ski2


See the analysis in `ai_output/gemini_ridge_invariance.md`.  This is the change Gemini suggested to make the filters transpose invariant.

The notebook ran with this fix in place (at commit `ea7afcbff`).

```diff
diff --git a/src/skimage/feature/corner.py b/src/skimage/feature/corner.py
index a50d01c2b..99d31d9df 100644
--- a/src/skimage/feature/corner.py
+++ b/src/skimage/feature/corner.py
@@ -215,10 +215,15 @@ def _hessian_matrix_with_gaussian(image, sigma=1, mode='reflect', cval=0, order=
     axes = range(ndim)
     if order == 'xy':
         axes = reversed(axes)
-    H_elems = [
-        gaussian_(gradients[ax0], order=orders[ax1])
-        for ax0, ax1 in combinations_with_replacement(axes, 2)
-    ]
+    H_elems = []
+    for ax0, ax1 in combinations_with_replacement(axes, 2):
+        if ax0 == ax1:
+            H_elems.append(gaussian_(gradients[ax0], order=orders[ax0]))
+        else:
+            # Symmetrize off-diagonals
+            h_ij = gaussian_(gradients[ax0], order=orders[ax1])
+            h_ji = gaussian_(gradients[ax1], order=orders[ax0])
+            H_elems.append(0.5 * (h_ij + h_ji))
     return H_elems
```

We also get transpose invariance from this source change, instead of the change above:

```diff
diff --git a/src/skimage/filters/ridges.py b/src/skimage/filters/ridges.py
index f43c5b12a..91791807d 100644
--- a/src/skimage/filters/ridges.py
+++ b/src/skimage/filters/ridges.py
@@ -82,7 +82,7 @@ def meijering(
     for sigma in sigmas:  # Filter for all sigmas.
         eigvals = hessian_matrix_eigvals(
             hessian_matrix(
-                image, sigma, mode=mode, cval=cval, use_gaussian_derivatives=True
+                image, sigma, mode=mode, cval=cval, use_gaussian_derivatives=False
             )
         )
         # Compute normalized eigenvalues l_i = e_i + sum_{j!=i} alpha * e_j.
@@ -158,7 +158,7 @@ def sato(image, sigmas=range(1, 10, 2), black_ridges=True, mode='reflect', cval=
     for sigma in sigmas:  # Filter for all sigmas.
         eigvals = hessian_matrix_eigvals(
             hessian_matrix(
-                image, sigma, mode=mode, cval=cval, use_gaussian_derivatives=True
+                image, sigma, mode=mode, cval=cval, use_gaussian_derivatives=False
             )
         )
         # Compute normalized tubeness (eqs. (9) and (22), ref. [1]_) as the
@@ -279,7 +279,7 @@ def frangi(
     for sigma in sigmas:  # Filter for all sigmas.
         eigvals = hessian_matrix_eigvals(
             hessian_matrix(
-                image, sigma, mode=mode, cval=cval, use_gaussian_derivatives=True
+                image, sigma, mode=mode, cval=cval, use_gaussian_derivatives=False
             )
         )
         # Sort eigenvalues by magnitude.
```

```diff
```

In [2]:
imgs = tpi.get_3d_images()
img = imgs[0]

In [3]:
img.shape

(20, 128, 128)

In [4]:
ridge_filts = [
    sfr.meijering,
    sfr.sato,
    sfr.frangi,
    sfr.hessian]

In [5]:
def func(img):
    return sfr.hessian(img)

ws_orig = func(img)

In [6]:
def check_func(img1, img2):
    assert np.allclose(img1, img2)

In [7]:
ws_rolled = tpi.rolled_proc(img, (2, 1, 0), func)
check_func(ws_orig, ws_rolled)

In [8]:
# All images are transpose invariant
tpi.assert_all_orders(imgs, func, chk_func=check_func)

Image 0
Ordering (0, 2, 1)
Ordering (1, 2, 0)
Ordering (2, 1, 0)
Ordering (2, 0, 1)
Ordering (1, 0, 2)
Image 1
Ordering (0, 2, 1)
Ordering (1, 2, 0)
Ordering (2, 1, 0)
Ordering (2, 0, 1)
Ordering (1, 0, 2)
Image 2
Ordering (0, 2, 1)
Ordering (1, 2, 0)
Ordering (2, 1, 0)
Ordering (2, 0, 1)
Ordering (1, 0, 2)
